# TRAM: SecBERT / RoBERTa retry (30 epochs, higher head learning rate)

Run the cells **top to bottom, one at a time**. Do not use "Run all" after editing.

Outputs go to `Exp_1_4_run_tram_2_epoc_30` (the old `run_tram_2_epoc_30` and `runs` folders are never touched). Every run also saves the model (`save_model=True`).

| Cell | What it does |
|---|---|
| 1 | Paths, imports, Drive mount |
| 2 | Load data + metric helpers |
| 3 | TF-IDF + SVM reference (seconds, CPU) |
| 4 | Defines `train_model` (nothing runs yet) |
| 5 | The 4 training runs (GPU: Runtime > Change runtime type > T4) |
| 6 | Results table |

## Section 1: Setup and Configurations
This section mounts Google Drive, imports necessary Python libraries, and sets up path constants and directory structures for saving the models and experimental outputs.

In [1]:
# Cell 1 - setup, paths, imports
import os, ast, json, random, warnings
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_DIR = "/content/drive/MyDrive/KDU/Thesis/Dataset"
OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30"   # NEW folder (old run_tram_2_epoc_30 and "runs" are untouched)
os.makedirs(OUT_ROOT, exist_ok=True)

MODELS = {"secbert": "jackaduma/SecBERT", "roberta": "roberta-base"}
warnings.filterwarnings("ignore", message="unknown class")
print("OUT_ROOT =", OUT_ROOT)

Mounted at /content/drive
OUT_ROOT = /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30


## Section 2: Data Loading and Metric Helpers
In this section, we load train, validation, and test splits, apply standard text-preprocessing, and construct the multilabel binarizer. We also define standard classification metrics (micro/macro F1, Precision, Recall, Hamming Loss) and ranking metrics (Hit@1, MRR@3) used to evaluate model performance.

In [2]:
# Cell 2 - data + metric helpers (same label space and metrics as the old notebook)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

def load_split(name):
    df = pd.read_csv(f"{DATA_DIR}/tram_{name}.tsv", sep="\t").rename(columns={"text1": "text"})
    df["y"] = df["labels"].map(lambda x: sorted(set(ast.literal_eval(x))))   # parse + de-duplicate labels
    return df[["text", "y"]]

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)       # train only; dev/test untouched
mlb = MultiLabelBinarizer().fit(train["y"])                                  # label space = labels seen in train
Y_train, Y_dev, Y_test = (mlb.transform(d["y"]) for d in (train, dev, test))
print(f"train={len(train)} dev={len(dev)} test={len(test)} classes={len(mlb.classes_)}")
assert len(train) == 3468, f"unexpected train size {len(train)}"

def micro_f1(y, p):
    tp = (y * p).sum(); fp = ((1 - y) * p).sum(); fn = (y * (1 - p)).sum()
    d = 2 * tp + fp + fn
    return 2.0 * tp / d if d > 0 else 0.0

def best_global_threshold(y, scores, grid):
    return float(grid[int(np.argmax([micro_f1(y, (scores >= t).astype(int)) for t in grid]))])

def apply_thresholds(scores, t):
    p = (scores >= t).astype(int)
    empty = p.sum(axis=1) == 0                       # never predict "no technique"
    p[np.where(empty)[0], scores[empty].argmax(axis=1)] = 1
    return p

def evaluate(y, p):
    present = y.sum(axis=0) > 0                      # macro-F1 only over classes present in this split
    return {
        "micro_f1": f1_score(y, p, average="micro", zero_division=0),
        "macro_f1": f1_score(y[:, present], p[:, present], average="macro", zero_division=0),
        "micro_precision": precision_score(y, p, average="micro", zero_division=0),
        "micro_recall": recall_score(y, p, average="micro", zero_division=0),
        "subset_accuracy": float((y == p).all(axis=1).mean()),
        "hamming_loss": hamming_loss(y, p),
        "n_classes_present": int(present.sum()),
    }

def ranking_metrics(y, scores):
    """hit@1 = top guess is a gold label. mrr@3 = 1/rank of first gold label in top 3 (0 if none).
    Rows whose gold labels are all unseen-in-train count as misses."""
    top3 = np.argsort(-scores, axis=1)[:, :3]
    hits = np.take_along_axis(y, top3, axis=1)       # rows x 3 of 0/1
    hit1 = hits[:, 0].mean()
    first = np.where(hits.any(axis=1), hits.argmax(axis=1) + 1, np.inf)
    return {"hit@1": float(hit1), "mrr@3": float(np.mean(1.0 / first))}

def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

train=3468 dev=604 test=725 classes=180


## Section 3: Baseline Reference (TF-IDF + SVM)
Here, we train a classic, lightweight baseline using a TF-IDF vectorizer coupled with a One-Vs-Rest Linear Support Vector Classifier (LinearSVC) on CPU. This provides a reference point for evaluating the performance of deep learning models.

In [3]:
# Cell 3 - TF-IDF + LinearSVC reference baseline (CPU, seconds)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train["text"])
Xtr, Xdv, Xte = (vec.transform(d["text"]) for d in (train, dev, test))
svm = OneVsRestClassifier(LinearSVC(C=1.0, random_state=42)).fit(Xtr, Y_train)
sc_dv, sc_te = svm.decision_function(Xdv), svm.decision_function(Xte)
t_svm = best_global_threshold(Y_dev, sc_dv, np.arange(-0.6, 0.31, 0.05))   # threshold picked on DEV only
svm_metrics = {**evaluate(Y_test, apply_thresholds(sc_te, t_svm)), **ranking_metrics(Y_test, sc_te)}
print(f"SVM threshold={t_svm:.2f}")
print({k: round(v, 4) for k, v in svm_metrics.items()})

SVM threshold=-0.45
{'micro_f1': 0.6961, 'macro_f1': 0.5427, 'micro_precision': 0.6722, 'micro_recall': 0.7218, 'subset_accuracy': 0.5752, 'hamming_loss': 0.0041, 'n_classes_present': 117, 'hit@1': 0.7421, 'mrr@3': 0.8138}


## Section 4: Transformer Training Logic
This section defines the datasets, PyTorch collate function, training/validation loops, and the main `train_model` function used to fine-tune our Hugging Face Transformer models (SecBERT and RoBERTa) with weighted cross-entropy losses, learning rate schedules, and early stopping.

In [4]:
# Cell 4 - transformer training (one clean train_model)
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

PROB_GRID = np.arange(0.1, 0.9, 0.05)

class TramDS(Dataset):
    def __init__(self, texts, Y, tok, max_length):
        self.enc = tok(list(texts), truncation=True, max_length=max_length)
        self.Y = Y
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.Y[i]
        return item

def make_collate(tok):
    def collate(batch):
        y = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float)
        enc = tok.pad([{k: v for k, v in b.items() if k != "labels"} for b in batch], return_tensors="pt")
        enc["labels"] = y
        return enc
    return collate

def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total, probs, labs = 0.0, [], []
    with torch.set_grad_enabled(training):
        for batch in tqdm(loader, disable=True):          # progress bars off -> clean logs
            batch = {k: v.to(device) for k, v in batch.items()}
            y = batch.pop("labels")
            logits = model(**batch).logits
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
            total += loss.item() * y.size(0)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            labs.append(y.cpu().numpy())
    return total / len(loader.dataset), np.concatenate(probs), np.concatenate(labs)

def train_model(model_name, output_dir, seed=42, epochs=30, patience=5, batch_size=16, lr=2e-5,
                head_lr=1e-3, max_length=128, warmup_ratio=0.1, use_pos_weight=True,
                pos_weight_cap=20.0, save_model=True):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_cls = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} classes={n_cls} epochs<={epochs} "
          f"lr={lr} head_lr={head_lr} pos_weight={use_pos_weight}")

    tok = AutoTokenizer.from_pretrained(model_name)
    col = make_collate(tok)
    tr_loader = DataLoader(TramDS(train["text"], Y_train, tok, max_length), batch_size=batch_size, shuffle=True, collate_fn=col)
    dv_loader = DataLoader(TramDS(dev["text"], Y_dev, tok, max_length), batch_size=batch_size * 2, collate_fn=col)
    te_loader = DataLoader(TramDS(test["text"], Y_test, tok, max_length), batch_size=batch_size * 2, collate_fn=col)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_cls, problem_type="multi_label_classification").to(device)

    pw = None
    if use_pos_weight:
        pos = np.clip(Y_train.sum(axis=0), 1, None)
        pw = torch.tensor(np.clip((len(Y_train) - pos) / pos, None, pos_weight_cap), dtype=torch.float, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

    head = [p for n, p in model.named_parameters() if n.startswith("classifier.")]
    body = [p for n, p in model.named_parameters() if not n.startswith("classifier.")]
    assert len(head) > 0, "no classifier.* parameters found"
    optimizer = torch.optim.AdamW([{"params": body, "lr": lr}, {"params": head, "lr": head_lr if head_lr else lr}])
    total_steps = len(tr_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_epoch, best_state, bad, history = -1.0, 0, None, 0, []
    for ep in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, tr_loader, loss_fn, device, optimizer, scheduler)
        dv_loss, dv_p, dv_y = run_epoch(model, dv_loader, loss_fn, device)
        m = evaluate(dv_y, apply_thresholds(dv_p, best_global_threshold(dv_y, dv_p, PROB_GRID)))
        history.append({"epoch": ep, "train_loss": tr_loss, "dev_loss": dv_loss, "dev_micro_f1": m["micro_f1"], "dev_macro_f1": m["macro_f1"]})
        print(f"  epoch {ep:2d}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, best_epoch, bad = m["micro_f1"], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop (no dev improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    _, dv_p, dv_y = run_epoch(model, dv_loader, loss_fn, device)
    thr = best_global_threshold(dv_y, dv_p, PROB_GRID)                 # threshold from DEV
    _, te_p, te_y = run_epoch(model, te_loader, loss_fn, device)
    metrics = {**evaluate(te_y, apply_thresholds(te_p, thr)), **ranking_metrics(te_y, te_p)}
    print("  TEST:", {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})

    np.save(f"{output_dir}/test_probs.npy", te_p); np.save(f"{output_dir}/dev_probs.npy", dv_p)
    np.save(f"{output_dir}/test_labels.npy", te_y); np.save(f"{output_dir}/dev_labels.npy", dv_y)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    json.dump({"model_name": model_name, "seed": seed, "best_epoch": best_epoch, "dev_best_micro_f1": best_f1,
               "threshold": thr, "test": metrics, "history": history,
               "config": dict(epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, head_lr=head_lr,
                              max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap)},
              open(f"{output_dir}/metrics.json", "w"), indent=2)
    if save_model:
        model.save_pretrained(output_dir); tok.save_pretrained(output_dir)
    print(f"  saved to {output_dir} (best epoch {best_epoch})")
    del model, optimizer, best_state
    import gc; gc.collect(); torch.cuda.empty_cache()
    return metrics

## Section 5: Executing Model Runs
This section runs the initial set of 30-epoch training configurations for both RoBERTa and SecBERT, checking both weighted (pos_weight) and unweighted loss configurations using seed 42.

In [5]:
# Cell 5 - the 4 runs (skips any run that already has model.safetensors, so you can resume after a disconnect)
print("OUT_ROOT =", OUT_ROOT)
assert OUT_ROOT.endswith("Exp_1_4_run_tram_2_epoc_30"), "OUT_ROOT is wrong - fix before training!"

RUNS = [  # (tag, use_pos_weight)
    ("roberta", True),  ("secbert", True),     # RoBERTa + class weights first (this is the model SHAP needs)
    ("roberta", False), ("secbert", False),
]
for tag, pw in RUNS:
    out = f"{OUT_ROOT}/{tag}_{'pw' if pw else 'nopw'}_s42"
    if os.path.exists(f"{out}/model.safetensors"):
        print("skip", out); continue
    print(f"\n>>> {tag}  use_pos_weight={pw}")
    train_model(MODELS[tag], out, seed=42, epochs=30, patience=5, lr=2e-5, head_lr=1e-3,
                use_pos_weight=pw, pos_weight_cap=20.0)

OUT_ROOT = /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30

>>> roberta  use_pos_weight=True
[roberta-base seed=42] device=cuda classes=180 epochs<=30 lr=2e-05 head_lr=0.001 pos_weight=True


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (22 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch  1: train_loss=0.2603 dev_loss=0.0356 dev_micro_f1=0.0369 dev_macro_f1=0.0006
  epoch  2: train_loss=0.0344 dev_loss=0.0337 dev_micro_f1=0.0461 dev_macro_f1=0.0029
  epoch  3: train_loss=0.0330 dev_loss=0.0314 dev_micro_f1=0.1995 dev_macro_f1=0.0338
  epoch  4: train_loss=0.0279 dev_loss=0.0245 dev_micro_f1=0.4014 dev_macro_f1=0.1344
  epoch  5: train_loss=0.0210 dev_loss=0.0195 dev_micro_f1=0.5158 dev_macro_f1=0.2562
  epoch  6: train_loss=0.0157 dev_loss=0.0168 dev_micro_f1=0.5924 dev_macro_f1=0.3460
  epoch  7: train_loss=0.0118 dev_loss=0.0151 dev_micro_f1=0.6265 dev_macro_f1=0.3853
  epoch  8: train_loss=0.0090 dev_loss=0.0144 dev_micro_f1=0.6488 dev_macro_f1=0.4197
  epoch  9: train_loss=0.0069 dev_loss=0.0138 dev_micro_f1=0.6671 dev_macro_f1=0.4508
  epoch 10: train_loss=0.0054 dev_loss=0.0137 dev_micro_f1=0.6724 dev_macro_f1=0.4852
  epoch 11: train_loss=0.0042 dev_loss=0.0141 dev_micro_f1=0.6643 dev_macro_f1=0.4862
  epoch 12: train_loss=0.0033 dev_loss=0.0138 dev_micr

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved to /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30/secbert_nopw_s42 (best epoch 16)


## Section 6: Summary of Phase 1 Model Configurations
This section compiles and prints a results table comparing the initial transformer configurations against the TF-IDF + SVM baseline using the validation split to select the optimal configuration.

In [6]:
# Cell 6 - results table (choose pw vs nopw by DEV, never by test)
rows = []
for tag in MODELS:
    for pw in ("pw", "nopw"):
        p = f"{OUT_ROOT}/{tag}_{pw}_s42/metrics.json"
        if os.path.exists(p):
            r = json.load(open(p)); t = r["test"]
            rows.append({"run": f"{tag}_{pw}", "best_epoch": r["best_epoch"], "dev_micro_f1": round(r["dev_best_micro_f1"], 4),
                         "test_micro_f1": round(t["micro_f1"], 4), "test_macro_f1": round(t["macro_f1"], 4),
                         "precision": round(t["micro_precision"], 4), "recall": round(t["micro_recall"], 4),
                         "hit@1": round(t["hit@1"], 4), "mrr@3": round(t["mrr@3"], 4)})
rows.append({"run": "TF-IDF+SVM (reference)", "best_epoch": "-", "dev_micro_f1": "-",
             "test_micro_f1": round(svm_metrics["micro_f1"], 4), "test_macro_f1": round(svm_metrics["macro_f1"], 4),
             "precision": round(svm_metrics["micro_precision"], 4), "recall": round(svm_metrics["micro_recall"], 4),
             "hit@1": round(svm_metrics["hit@1"], 4), "mrr@3": round(svm_metrics["mrr@3"], 4)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
for tag in MODELS:
    sub = res[res["run"].isin([f"{tag}_pw", f"{tag}_nopw"])]
    if len(sub) == 2:
        print(f"\n{tag}: better on DEV -> {sub.loc[sub['dev_micro_f1'].astype(float).idxmax(), 'run']}")
res

                   run best_epoch dev_micro_f1  test_micro_f1  test_macro_f1  precision  recall  hit@1  mrr@3
            secbert_pw         18       0.6993         0.7042         0.5363     0.7245  0.6849 0.7434 0.8055
          secbert_nopw         16        0.697         0.6888         0.5173     0.7119  0.6671 0.7338 0.7977
            roberta_pw         25        0.736         0.7077         0.5681     0.7268  0.6897 0.7572 0.8113
          roberta_nopw         19        0.736         0.7124         0.5810     0.7423  0.6849 0.7614 0.8138
TF-IDF+SVM (reference)          -            -         0.6961         0.5427     0.6722  0.7218 0.7421 0.8138

secbert: better on DEV -> secbert_pw

roberta: better on DEV -> roberta_pw


,run,best_epoch,dev_micro_f1,test_micro_f1,test_macro_f1,precision,recall,hit@1,mrr@3
0,secbert_pw,18,0.6993,0.7042,0.5363,0.7245,0.6849,0.7434,0.8055
1,secbert_nopw,16,0.697,0.6888,0.5173,0.7119,0.6671,0.7338,0.7977
2,roberta_pw,25,0.736,0.7077,0.5681,0.7268,0.6897,0.7572,0.8113
3,roberta_nopw,19,0.736,0.7124,0.5810,0.7423,0.6849,0.7614,0.8138
4,TF-IDF+SVM (reference),-,-,0.6961,0.5427,0.6722,0.7218,0.7421,0.8138


## Section 7: Reproducibility Checks (Additional Seeds)
To verify statistical significance and reliability, we run our best-performing configurations on two additional random seeds (43 and 44).

In [7]:
# Cell 7 - 2 more seeds (43, 44), class weights ON, both models
print("OUT_ROOT =", OUT_ROOT)
assert OUT_ROOT.endswith("Exp_1_4_run_tram_2_epoc_30"), "OUT_ROOT is wrong - fix before training!"
for seed in (43, 44):
    for tag in ("roberta", "secbert"):
        out = f"{OUT_ROOT}/{tag}_pw_s{seed}"
        if os.path.exists(f"{out}/model.safetensors"):
            print("skip", out); continue
        print(f"\n>>> {tag} seed={seed}")
        train_model(MODELS[tag], out, seed=seed, epochs=30, patience=5, lr=2e-5, head_lr=1e-3,
                    use_pos_weight=True, pos_weight_cap=20.0)

OUT_ROOT = /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30

>>> roberta seed=43
[roberta-base seed=43] device=cuda classes=180 epochs<=30 lr=2e-05 head_lr=0.001 pos_weight=True


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch  1: train_loss=0.4284 dev_loss=0.2465 dev_micro_f1=0.2703 dev_macro_f1=0.0918
  epoch  2: train_loss=0.1872 dev_loss=0.1573 dev_micro_f1=0.4587 dev_macro_f1=0.2491
  epoch  3: train_loss=0.1185 dev_loss=0.1262 dev_micro_f1=0.5168 dev_macro_f1=0.3340
  epoch  4: train_loss=0.0823 dev_loss=0.1158 dev_micro_f1=0.5851 dev_macro_f1=0.4329
  epoch  5: train_loss=0.0552 dev_loss=0.1171 dev_micro_f1=0.6153 dev_macro_f1=0.4829
  epoch  6: train_loss=0.0400 dev_loss=0.1112 dev_micro_f1=0.6365 dev_macro_f1=0.5369
  epoch  7: train_loss=0.0302 dev_loss=0.1253 dev_micro_f1=0.6694 dev_macro_f1=0.5490
  epoch  8: train_loss=0.0224 dev_loss=0.1270 dev_micro_f1=0.6899 dev_macro_f1=0.5648
  epoch  9: train_loss=0.0165 dev_loss=0.1489 dev_micro_f1=0.6963 dev_macro_f1=0.5684
  epoch 10: train_loss=0.0132 dev_loss=0.1475 dev_micro_f1=0.6862 dev_macro_f1=0.5715
  epoch 11: train_loss=0.0099 dev_loss=0.1398 dev_micro_f1=0.7047 dev_macro_f1=0.5861
  epoch 12: train_loss=0.0085 dev_loss=0.1625 dev_micr

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved to /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30/roberta_pw_s43 (best epoch 13)

>>> secbert seed=43
[jackaduma/SecBERT seed=43] device=cuda classes=180 epochs<=30 lr=2e-05 head_lr=0.001 pos_weight=True


... (11 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch  1: train_loss=0.4604 dev_loss=0.2986 dev_micro_f1=0.1281 dev_macro_f1=0.0248
  epoch  2: train_loss=0.2472 dev_loss=0.1894 dev_micro_f1=0.4002 dev_macro_f1=0.1849
  epoch  3: train_loss=0.1418 dev_loss=0.1287 dev_micro_f1=0.5297 dev_macro_f1=0.3134
  epoch  4: train_loss=0.0797 dev_loss=0.1125 dev_micro_f1=0.5991 dev_macro_f1=0.3919
  epoch  5: train_loss=0.0472 dev_loss=0.1048 dev_micro_f1=0.6382 dev_macro_f1=0.4715
  epoch  6: train_loss=0.0313 dev_loss=0.1124 dev_micro_f1=0.6524 dev_macro_f1=0.4850
  epoch  7: train_loss=0.0223 dev_loss=0.1200 dev_micro_f1=0.6724 dev_macro_f1=0.5173
  epoch  8: train_loss=0.0166 dev_loss=0.1248 dev_micro_f1=0.6727 dev_macro_f1=0.5261
  epoch  9: train_loss=0.0129 dev_loss=0.1367 dev_micro_f1=0.7025 dev_macro_f1=0.5457
  epoch 10: train_loss=0.0102 dev_loss=0.1456 dev_micro_f1=0.6894 dev_macro_f1=0.5392
  epoch 11: train_loss=0.0081 dev_loss=0.1512 dev_micro_f1=0.7005 dev_macro_f1=0.5581
  epoch 12: train_loss=0.0066 dev_loss=0.1557 dev_micr

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved to /content/drive/MyDrive/KDU/Thesis/Exp_1_4_run_tram_2_epoc_30/secbert_pw_s44 (best epoch 23)


## Section 8: Aggregated Performance Metrics
This section aggregates results across all random seeds (42, 43, 44) to calculate and present the mean and standard deviation for our evaluation metrics.

In [8]:
# Cell 8 - mean ± std over seeds 42, 43, 44 (sample std, ddof=1)
SEEDS = (42, 43, 44)
keys = ["micro_f1", "macro_f1", "micro_precision", "micro_recall", "hit@1", "mrr@3"]
rows, per_seed = [], []
for tag in MODELS:
    vals = {k: [] for k in keys}; used = []
    for s in SEEDS:
        p = f"{OUT_ROOT}/{tag}_pw_s{s}/metrics.json"
        if os.path.exists(p):
            t = json.load(open(p))["test"]; used.append(s)
            for k in keys: vals[k].append(t[k])
            per_seed.append({"model": tag, "seed": s, **{k: round(t[k], 4) for k in keys}})
    if len(used) >= 2:
        rows.append({"model": tag, "seeds": len(used),
                     **{k: f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}" for k, v in vals.items()}})
rows.append({"model": "TF-IDF+SVM (1 run)", "seeds": "-", **{k: f"{svm_metrics[k]:.3f}" for k in keys}})
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer-seed values:")
print(pd.DataFrame(per_seed).to_string(index=False))

             model seeds      micro_f1      macro_f1 micro_precision  micro_recall         hit@1         mrr@3
           secbert     3 0.703 ± 0.001 0.543 ± 0.008   0.712 ± 0.013 0.694 ± 0.010 0.743 ± 0.004 0.805 ± 0.004
           roberta     3 0.705 ± 0.011 0.563 ± 0.013   0.721 ± 0.017 0.691 ± 0.007 0.752 ± 0.007 0.809 ± 0.007
TF-IDF+SVM (1 run)     -         0.696         0.543           0.672         0.722         0.742         0.814

Per-seed values:
  model  seed  micro_f1  macro_f1  micro_precision  micro_recall  hit@1  mrr@3
secbert    42    0.7042    0.5363           0.7245        0.6849 0.7434 0.8055
secbert    43    0.7022    0.5514           0.6993        0.7051 0.7476 0.8080
secbert    44    0.7028    0.5413           0.7127        0.6932 0.7393 0.8000
roberta    42    0.7077    0.5681           0.7268        0.6897 0.7572 0.8113
roberta    43    0.6931    0.5487           0.7016        0.6849 0.7434 0.8002
roberta    44    0.7154    0.5735           0.7338        0.6980

## Section 9: Benchmarking Against Published Literature
Finally, we evaluate the models using specialized paper-style Precision@1 and Recall@1 metrics to compare our results directly against baseline architectures reported in literature.

In [9]:

# Cell 9 - paper-style micro P@1 / R@1 (+ the older sample-averaged R@1 for comparison)
gold_full = np.array([len(y) for y in test["y"]])      # ALL gold labels per row, incl. labels unseen in train
N = len(test); total_gold = gold_full.sum()

def p1_r1(scores):
    top1 = scores.argmax(axis=1)
    hit = Y_test[np.arange(N), top1]                   # 1 if top guess is a gold label
    return {"P@1 (micro)": hit.mean(),
            "R@1 (micro)": hit.sum() / total_gold,     # paper-style: hits / ALL gold labels
            "R@1 (sample-avg)": (hit / gold_full).mean()}   # older formula, NOT comparable to the paper

rows = [{"model": "TF-IDF+SVM", **p1_r1(sc_te)}]
for tag in MODELS:
    per = []
    for s in (42, 43, 44):
        p = f"{OUT_ROOT}/{tag}_pw_s{s}/test_probs.npy"
        if os.path.exists(p):
            per.append(p1_r1(np.load(p)))
    if per:
        rows.append({"model": f"{tag} (mean of {len(per)} seeds)", **{k: np.mean([d[k] for d in per]) for k in per[0]}})
rows.append({"model": "Paper: Asymmetric NCE (as reported)", "P@1 (micro)": .770, "R@1 (micro)": .658, "R@1 (sample-avg)": np.nan})
rows.append({"model": "Paper: XR-Linear (as reported)", "P@1 (micro)": .674, "R@1 (micro)": .626, "R@1 (sample-avg)": np.nan})
print(f"rows={N}, total gold labels={total_gold}, avg labels/row={total_gold/N:.3f}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

rows=725, total gold labels=851, avg labels/row=1.174
                              model  P@1 (micro)  R@1 (micro)  R@1 (sample-avg)
                         TF-IDF+SVM        0.742        0.632             0.688
          secbert (mean of 3 seeds)        0.743        0.633             0.693
          roberta (mean of 3 seeds)        0.752        0.640             0.701
Paper: Asymmetric NCE (as reported)        0.770        0.658               NaN
     Paper: XR-Linear (as reported)        0.674        0.626               NaN
